In [1]:
import pickle
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import mmh3
import os
import pickle
def get_timestamp_us(mstr):
    _i, _n = mstr.split(".")
    while len(_n) != 6:
        _n = _n + "0"
    return int(_i) * 1e6 + int(_n)

In [2]:
class Chronus:
    
    def __init__(self, w, m):
        self.m = m
        self.w = w
        self.rtt_cnt = np.zeros(shape = (self.m), dtype = np.uint64)
        self.IDs = [None for i in range(self.m)]
        self.gen_next_seq = np.zeros(shape = (self.w), dtype = np.uint32)
        self.gen_time = np.zeros(shape = (self.w), dtype = np.uint64)
        self.gen_IDs = [None for i in range(self.w)]
        self.mseeds = np.random.randint(10000, 20000)
        self.est_rtt_flows = set([])
        self.gen_seed = np.random.randint(10000, 20000)
        self.fp_seed = np.random.randint(10000, 20000)
        self.d = 3
        self.timeout = None
    
    def set_timeout(self, T):
        self.timeout = T
    
    def insert(self, src, dst, sport, dport, seq, ack, _time):
        hash_key = None
        if src < dst:
            hash_key = "{} {} {} {}".format(src, sport, dst, dport)
        else:
            hash_key = "{} {} {} {}".format(dst, dport, src, sport)
        hash_idx = mmh3.hash(hash_key, seed = self.gen_seed, signed = False) % self.w
        rtt_inc = None
        fp = mmh3.hash(hash_key, seed = self.fp_seed, signed = False) % ((1<<16)-1)
        # 触发机制
        if self.gen_IDs[hash_idx] == hash_key:
            if self.gen_next_seq[hash_idx] == seq:
                rtt_inc = _time - self.gen_time[hash_idx]
                if rtt_inc > self.timeout or rtt_inc < 0:
                    rtt_inc = None
            self.gen_next_seq[hash_idx] = ack
            self.gen_time[hash_idx] = _time
        elif self.gen_IDs[hash_idx] == None:
            self.gen_IDs[hash_idx] = hash_key
            self.gen_next_seq[hash_idx] = ack
            self.gen_time[hash_idx] = _time
        else:
            if _time - self.gen_time[hash_idx] > self.timeout:
                self.gen_IDs[hash_idx] = hash_key
                self.gen_next_seq[hash_idx] = ack
                self.gen_time[hash_idx] = _time
        if rtt_inc != None:
            empty_idx = None
            min_idx = None
            min_val = None
            for i in range(self.d):
                idx = mmh3.hash(hash_key + str(i), seed = self.mseeds, signed = False) % self.m
                if self.IDs[idx] == None:
                    empty_idx = idx
                elif self.IDs[idx] == hash_key:
                    self.rtt_cnt[idx] += rtt_inc
                    return
                else:
                    if min_idx == None:
                        min_idx = idx
                        min_val = self.rtt_cnt[idx]
                    else:
                        if self.rtt_cnt[idx] < min_val:
                            min_idx = idx
                            min_val = self.rtt_cnt[idx]
            if empty_idx != None:
                self.IDs[empty_idx] = hash_key
                self.rtt_cnt[empty_idx] += rtt_inc
            else:
                self.rtt_cnt[min_idx] += rtt_inc
                if np.random.random() <= (rtt_inc / (rtt_inc + self.rtt_cnt[min_idx])):
                    self.IDs[min_idx] = hash_key
    
    def detect(self, T):
        for i in range(self.m):
            if self.rtt_cnt[i] >= T:
                self.est_rtt_flows.add(self.IDs[i])
    
    def estimate(self, key):
        for i in range(self.d):
            hash_idx = mmh3.hash(key + str(i), seed = self.mseeds, signed = False) % self.m
            if self.IDs[hash_idx] == key:
                return self.rtt_cnt[hash_idx]
        return 1

In [3]:
def get_performance(real_set, pred_set):
    TP = len(pred_set.intersection(real_set))
    FP = len(pred_set - real_set)
    FN = len(real_set - pred_set)
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    F1 = (2 * precision * recall) / (precision + recall)
    return precision, recall, F1

In [4]:
f_r = open("quantiles_IMC.txt", 'r')
quantile_data = f_r.readlines()
f_r.close()
quantile_60_dict = defaultdict(int)
quantile_70_dict = defaultdict(int)
quantile_80_dict = defaultdict(int)
quantile_90_dict = defaultdict(int)
quantile_99_dict = defaultdict(int)
for row in quantile_data:
    name, x1, y1, x2, y2, x3, y3, x4, y4, x5, y5 = row.split("-")
    quantile_60_dict[name] = int(float(y1))
    quantile_70_dict[name] = int(float(y2))
    quantile_80_dict[name] = int(float(y3))
    quantile_90_dict[name] = int(float(y4))
    quantile_99_dict[name] = int(float(y5))

In [5]:
f_r = open("thresholds_60_imc.txt")
threshold_60_datas = f_r.readlines()
f_r.close()
threshold_60_dict = defaultdict(int)
for row in threshold_60_datas:
    name, thr = row.split("-")
    thr = int(float(thr))
    threshold_60_dict[name] = thr
f_r = open("thresholds_70_imc.txt")
threshold_70_datas = f_r.readlines()
f_r.close()
threshold_70_dict = defaultdict(int)
for row in threshold_70_datas:
    name, thr = row.split("-")
    thr = int(float(thr))
    threshold_70_dict[name] = thr
f_r = open("thresholds_80_imc.txt")
threshold_80_datas = f_r.readlines()
f_r.close()
threshold_80_dict = defaultdict(int)
for row in threshold_80_datas:
    name, thr = row.split("-")
    thr = int(float(thr))
    threshold_80_dict[name] = thr
f_r = open("thresholds_90_imc.txt")
threshold_90_datas = f_r.readlines()
f_r.close()
threshold_90_dict = defaultdict(int)
for row in threshold_90_datas:
    name, thr = row.split("-")
    thr = int(float(thr))
    threshold_90_dict[name] = thr
f_r = open("thresholds_99_imc.txt")
threshold_99_datas = f_r.readlines()
f_r.close()
threshold_99_dict = defaultdict(int)
for row in threshold_99_datas:
    name, thr = row.split("-")
    thr = int(float(thr))
    threshold_99_dict[name] = thr

In [6]:
def get_timeout(qua, name):
    if qua == 60:
        return quantile_60_dict[name]
    elif qua == 70:
        return quantile_70_dict[name]
    elif qua == 80:
        return quantile_80_dict[name]
    elif qua == 90:
        return quantile_90_dict[name]
    elif qua == 99:
        return quantile_99_dict[name]

def get_threshold(qua, name):
    if qua == 60:
        return threshold_60_dict[name]
    elif qua == 70:
        return threshold_70_dict[name]
    elif qua == 80:
        return threshold_80_dict[name]
    elif qua == 90:
        return threshold_90_dict[name]
    elif qua == 99:
        return threshold_99_dict[name]

In [7]:
f_r = open("./start_time_imc.pkl", 'rb')
start_time_dict = pickle.load(f_r)
f_r.close()
data_dir = ['IMC10/']
mem_lst = [64, 128, 256, 512, 1024]
qua_lst = [80, 90, 99]
for _dir in data_dir:
    for qua in qua_lst:
        f_w = open("chronus_{}_{}.txt".format(_dir.strip("/"), qua), 'w')
        for mem in mem_lst:
            precision_lst = []
            recall_lst    = []
            F1_lst        = []
            ARE_lst       = []
            for name in os.listdir(_dir):
                if name.find("point") != -1:
                    continue
                print(str(qua)+ ":" + _dir + name, " is running...")
                timeout = get_timeout(qua, _dir + name)
                detect_thr = get_threshold(qua, _dir + name)
                print("timeout:", timeout, ", detection threshold:", detect_thr)
                start_time = start_time_dict[_dir + name]
                ratio = 0.5
                w = int(mem * ratio * 1024 / 10)
                m = int(mem * (1 - ratio) * 1024 / 16)
                sketch = Chronus(w, m)
                sketch.set_timeout(timeout)
                f = open(_dir + name, 'r')
                datas = f.readlines()
                f.close()
                flow_ack_dict = defaultdict(int)
                flow_time_dict = defaultdict(int)
                flow_rtt_dict = defaultdict(int)
                for pkt in tqdm(datas):
                    src, dst, sport, dport, seq, ack, _tag, _t = pkt.split()
                    if _t == "time":
                        continue
                    seq = int(seq)
                    ack = int(ack)
                    if seq == 0 or ack == 0:
                        continue
                    _time = get_timestamp_us(_t) - start_time + 1
                    hash_key = None
                    if src < dst:
                        hash_key = "{} {} {} {}".format(src, sport, dst, dport)
                    else:
                        hash_key = "{} {} {} {}".format(dst, dport, src, sport)
                    if hash_key not in flow_ack_dict:
                        flow_ack_dict[hash_key] = ack
                        flow_time_dict[hash_key] = _time
                    else:
                        if flow_ack_dict[hash_key] != seq:
                            flow_ack_dict[hash_key] = ack
                            flow_time_dict[hash_key] = _time
                        else:
                            # bounded time delay
                            if 0<= _time - flow_time_dict[hash_key] <= timeout:
                                flow_rtt_dict[hash_key] += _time - flow_time_dict[hash_key]
                            flow_ack_dict[hash_key] = ack
                            flow_time_dict[hash_key] = _time
                    sketch.insert(src, dst, sport, dport, seq, ack, _time)
                sketch.est_rtt_flows = set([])
                sketch.detect(detect_thr)
                pred_set = sketch.est_rtt_flows
                _are = 0.0
                _cnt = 0
                real_set = set([])
                for key in flow_rtt_dict:
                    if flow_rtt_dict[key] >= detect_thr:
                        real_set.add(key)
                        #print(sketch.estimate(key))
                        _are += abs(flow_rtt_dict[key] - sketch.estimate(key)) / flow_rtt_dict[key]
                        _cnt += 1
                _are = _are / _cnt
                ARE_lst.append(_are)
                p, r, f1 = get_performance(real_set, pred_set)
                print("[瞬时值]{}KB P: {}, R: {}, F1: {}, ARE: {}\n".format(mem, p, r, f1, _are))
                precision_lst.append(p)
                recall_lst.append(r)
                F1_lst.append(f1)
            avg_precision = sum(precision_lst) / len(precision_lst)
            avg_recall = sum(recall_lst) / len(recall_lst)
            avg_f1_score = sum(F1_lst) / len(F1_lst)
            avg_ARE = sum(ARE_lst) / len(ARE_lst)
            print("[平均值]{}KB P: {}, R: {}, F1: {}, ARE: {}\n".format(mem, avg_precision, avg_recall, avg_f1_score, avg_ARE))
            f_w.write("{}KB P: {}, R: {}, F1: {}, ARE: {}\n".format(mem, avg_precision, avg_recall, avg_f1_score, avg_ARE))
        f_w.close()

80:IMC10/IMC10.txt  is running...
timeout: 5591 , detection threshold: 1596782


100%|██████████| 14720318/14720318 [02:46<00:00, 88245.61it/s] 


[瞬时值]64KB P: 0.9339622641509434, R: 0.99, F1: 0.9611650485436893, ARE: 0.023732673685878978

[平均值]64KB P: 0.9339622641509434, R: 0.99, F1: 0.9611650485436893, ARE: 0.023732673685878978

80:IMC10/IMC10.txt  is running...
timeout: 5591 , detection threshold: 1596782


100%|██████████| 14720318/14720318 [03:07<00:00, 78329.14it/s]


[瞬时值]128KB P: 0.9801980198019802, R: 0.99, F1: 0.9850746268656716, ARE: 0.019913269339414005

[平均值]128KB P: 0.9801980198019802, R: 0.99, F1: 0.9850746268656716, ARE: 0.019913269339414005

80:IMC10/IMC10.txt  is running...
timeout: 5591 , detection threshold: 1596782


...

[瞬时值]256KB P: 0.9801980198019802, R: 0.99, F1: 0.9850746268656716, ARE: 0.008703798814799471

[平均值]256KB P: 0.9801980198019802, R: 0.99, F1: 0.9850746268656716, ARE: 0.008703798814799471

99:IMC10/IMC10.txt  is running...
timeout: 794774 , detection threshold: 59905619


100%|██████████| 14720318/14720318 [02:51<00:00, 85820.09it/s]


[瞬时值]512KB P: 1.0, R: 1.0, F1: 1.0, ARE: 0.006065135196335011

[平均值]512KB P: 1.0, R: 1.0, F1: 1.0, ARE: 0.006065135196335011

99:IMC10/IMC10.txt  is running...
timeout: 794774 , detection threshold: 59905619


100%|██████████| 14720318/14720318 [02:51<00:00, 85947.31it/s] 


[瞬时值]1024KB P: 1.0, R: 1.0, F1: 1.0, ARE: 0.002369738526445963

[平均值]1024KB P: 1.0, R: 1.0, F1: 1.0, ARE: 0.002369738526445963

